# A5 — Tier 2 trên toàn bộ 307 cửa sổ (Phước)

Mục tiêu D3: **A5a = dry run 10 cửa sổ có resume**, rồi **A5b = full 307 cửa sổ** → `results/verify/<video_id>/window_NNNN.json`.
Đây là hạng mục **bắt buộc của đề tài** (tier 2 = VLM sinh ngôn ngữ thật) và là mảng duy nhất chưa chạy end-to-end.

Điều kiện: merge `main` đã có `scripts/run_verify.py` (A3, PR #27) + `VLMVerifier` (A2, PR #16).

**Chưa có trong A3 (A3b của Huy, chờ quyết định D-1):** chunking cửa sổ > 30 s, ghi `elapsed_s`/median-p90, `--shard i/N`.
Nếu A3b đã lên `main` trước khi bạn chạy A5b → `git pull` rồi chạy, đừng chạy full bằng bản cũ (protocol đổi = chạy lại).

Runtime: **T4 GPU** (Change runtime type). Kết quả ghi thẳng vào **Drive** để mất session không mất tiến độ (runner có resume).

In [ ]:
# Cell 1 — GPU. Không phải T4 -> Runtime > Change runtime type > T4 GPU.
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Cần GPU T4"
assert torch.cuda.get_device_name(0).startswith("Tesla T4"), f"T4 được yêu cầu, thấy {torch.cuda.get_device_name(0)}"

In [ ]:
# Cell 2 — clone đúng main đã merge A3 + C7, cài deps VLM (GIỮ torch của Colab).
import os

REPO = "/content/vlm-anomalies-detection"
if not os.path.isdir(REPO):
    !git clone https://github.com/Kyosaki6/vlm-anomalies-detection.git {REPO}
os.chdir(REPO)
!git pull --ff-only
!git log --oneline -3
!test -f scripts/run_verify.py || (echo 'THIEU run_verify.py — main chua merge A3 (M1a)'; raise SystemExit(1))

!pip install -q -r requirements-vlm.txt
!pip freeze | grep -iE "^(torch|transformers|bitsandbytes|accelerate)=="

In [ ]:
# Cell 3 — weight VLM trước (~6.5 GB, KHÔNG tính vào thời gian A5).
!huggingface-cli download Qwen/Qwen2.5-VL-3B-Instruct > /dev/null && echo "weights OK"

In [ ]:
# Cell 4 — 114 video CÓ cửa sổ (4.52 GB) = tập con của 290 video test.
# Downloader không có --only, nên tải full (8.7 GB, resumable, file có sẵn là bỏ qua).
# Đĩa Colab ~78 GB: OK. Nếu chật: xoá /content/sample_data và cache pip trước.
import json

need = sorted({w["video_id"] for w in json.load(open("results/windows/_index.json"))["windows"]})
print("videos can windows:", len(need))
!df -h /content | tail -1
!python scripts/download_ucf_crime.py --dry-run | tail -3   # xem dung lượng trước khi tải

In [ ]:
# Cell 5 — TẢI VIDEO (chạy nền cho đỡ chờ; resumable: chạy lại = tiếp tục).
!python scripts/download_ucf_crime.py --threads 8

import csv
rows = list(csv.DictReader(open("manifests/ucf_test.csv", encoding="utf-8-sig")))
missing = [r["video_id"] for r in rows if r["video_id"] in set(need) and not os.path.exists(r["path"])]
print("thieu video co cua so:", len(missing), missing[:10])
assert not missing, f"Thieu {len(missing)} video — không được chạy A5b khi thiếu (fail loud, không skip)"

In [ ]:
# Cell 6 — Drive: --out-dir đặt TRÊN DRIVE để session chết không mất file.
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/vlm_a5/verify"
import os
os.makedirs(OUT, exist_ok=True)
print("OUT =", OUT)

In [ ]:
# Cell 7 — DRY RUN: đếm cửa sổ, KHÔNG load model, không ghi gì.
# Kỳ vọng: "307 window xét".
!python scripts/run_verify.py --manifest manifests/ucf_test.csv \
    --windows-dir results/windows --out-dir {OUT} --k 4 --dry-run | tail -5

## A5a — pilot 10 cửa sổ (thật)

Chạy **in-process** (gọi `main(argv)`) chứ không qua `!python` để đo được `torch.cuda.max_memory_allocated()`.
Runner chưa ghi `elapsed_s` (việc của A3b) nên s/window tính từ mtime của file output.

In [ ]:
# Cell 8 — A5a: 10 cửa sổ thật + đo VRAM/s/window.
import json, statistics, sys, time
from pathlib import Path

sys.path.insert(0, ".")
import torch
from scripts.run_verify import main

OUT = "/content/drive/MyDrive/vlm_a5/verify"
argv = ["--manifest", "manifests/ucf_test.csv", "--windows-dir", "results/windows",
        "--out-dir", OUT, "--k", "4", "--continue-on-error",
        "--verifier-model", "Qwen/Qwen2.5-VL-3B-Instruct",
        "--limit", "10"]

torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
rc = main(argv)
elapsed = time.perf_counter() - t0

files = sorted(Path(OUT).rglob("window_*.json"))
mt = [f.stat().st_mtime for f in files]
delta = [b - a for a, b in zip(mt, mt[1:])]
pilot = {
    "rc": rc, "n_files": len(files), "elapsed_s": round(elapsed, 1),
    "median_s_per_window": round(statistics.median(delta), 1) if delta else None,
    "peak_alloc_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
    "peak_reserved_gb": round(torch.cuda.max_memory_reserved() / 2**30, 2),
}
print("PILOT", json.dumps(pilot))
assert rc in (0, 2), f"run_verify lỗi: {rc}"
assert len(files) >= 10, f"mới {len(files)} file, kỳ vọng 10"
# A1 đo 4.45 s/window trên T4 -> pilot trong khoảng 2x là OK
assert pilot["median_s_per_window"] and pilot["median_s_per_window"] < 9, "chậm bất thường"
assert pilot["peak_alloc_gb"] < 5.5, "peak VRAM vượt ngưỡng A1 (5.5 GB)"

In [ ]:
# Cell 9 — CHỨNG MINH RESUME: chạy lại y hệt -> không verify lại cửa sổ nào.
before = len(list(Path(OUT).rglob("window_*.json")))
rc = main(argv)   # cùng argv, đã có file
after = len(list(Path(OUT).rglob("window_*.json")))
print(f"files {before} -> {after} | rc={rc} (kỳ vọng: after == before, rc=0)")
assert after == before, "resume không hoạt động: file bị ghi lại"

## A5b — full 307 cửa sổ

≈ 480 chunk-call (nếu A3b/D-1 đã có) ≈ **36 phút T4** + ~2.5 phút load model. Session chết → chạy lại cell sau, runner tự skip file đã xong.

- **Chạy bằng CLI** (không in-process) để output ra log thật, chứa `xong: N window xét...`.
- Nếu quota fail: Huy chạy `--shard 2/2` trên tài khoản Colab thứ hai (A3b).

In [ ]:
# Cell 10 — FULL RUN. Chạy lại cell này bao nhiêu lần tùy ý (resume).
!python scripts/run_verify.py --manifest manifests/ucf_test.csv \
    --windows-dir results/windows --out-dir {OUT} --k 4 \
    --verifier-model Qwen/Qwen2.5-VL-3B-Instruct --continue-on-error 2>&1 | tee /content/drive/MyDrive/vlm_a5/run_a5b.log

In [ ]:
# Cell 11 — NGHIỆM THU A5b: đủ 307 file, log chi phí (điều kiện nghiệm thu của A5).
import json, statistics
from pathlib import Path

files = sorted(Path(OUT).rglob("window_*.json"))
idx = json.load(open("results/windows/_index.json"))["windows"]
print(f"verify files: {len(files)} / {len(idx)} window")

mt = [f.stat().st_mtime for f in files]
delta = [b - a for a, b in zip(mt, mt[1:])]
if delta:
    s = sorted(delta)
    print("s/window  median", round(statistics.median(s), 1),
          "p90", round(s[int(0.9 * len(s))], 1))
print("peak_alloc_gb", round(torch.cuda.max_memory_allocated() / 2**30, 2))

parse_fail = 0
confirmed = 0
for f in files:
    d = json.loads(f.read_text(encoding="utf-8-sig"))
    vr = d.get("verify_results") or [d]  # layout do A3b/C7a chốt
    for v in vr:
        if v.get("is_anomaly") is True:
            confirmed += 1
print(f"VLM confirmed: {confirmed}/{len(files)} = {100*confirmed/max(1,len(files)):.1f}% escalated")
assert len(files) == len(idx), f"THIEU {len(idx)-len(files)} window — A5b CHUA xong"

In [ ]:
# Cell 12 — đưa kết quả về máy (zip -> tải về, hoặc commit lên nhánh nếu < 25 MB).
import shutil
shutil.make_archive("/content/verify_a5b", "zip", OUT)
print("-> /content/verify_a5b.zip")
from google.colab import files as colab_files
colab_files.download("/content/verify_a5b.zip")

## Bẫy đã biết

- **`confidence` của Qwen 3B gần như hằng số ~0.95** → không được present nó là tín hiệu (`min_confidence` 0.5 trong `policy.yaml` là vô hiệu).
- **`event_type` từ VLM bias nặng về Robbery** (0/8 đúng ở sanity) → alert lấy type từ **typer C3** (quyết định D-2); VLM chỉ quyết `is_anomaly` + caption.
- Parse hỏng 2 lần → `VLMVerifier` **raise** (không bịa kết quả); `--continue-on-error` để 1 cửa sổ lỗi không kill cả batch, exit code 2 = có lỗi.
- Không bao giờ ghi output vào `results/verify/` bằng `--fake` (script tự chặn); output thật phải về `results/verify/` trong repo.
- Cửa sổ dài (65/307 > 30 s, max 490 s): chờ **D-1/A3b** (chunk ≤ 30 s, OR verdict). Chạy full bằng bản chưa chunk = phải chạy lại.

## Sau khi A5b xong

1. Tải `verify/` vào repo → `results/verify/`.
2. **A6a**: rút mẫu 20 cửa sổ (10 confirmed / 10 rejected, seed cố định) vào `notes/a6-handcheck-sample.csv` TRƯỚC khi đọc caption.
3. báo Thịnh để chạy `build_results.py --verify-dir results/verify` (C7d) → dashboard có caption/alert thật.